# LGMD-on-SpLiCE: Small-Scale Proof of Concept

Replaces LGMD's expensive per-class LLM vocabulary + red-circle CLIP occlusion with SpLiCE's sparse, non-negative, language-grounded decomposition of grid-cropped CLIP embeddings. See `/home/.../sparse_concept_discovery` plan for full design rationale.

Runs on 3 Imagenette classes end-to-end: encoder activations -> SpLiCE semantic coefficients (S) -> learn concept basis (W) -> reconstruct held-out activations -> evaluate predictive preservation -> visualize concept heatmaps.

In [ ]:
# In Colab: upload/sync this project folder first (e.g. via Google Drive mount,\n# `git clone` your own remote for this repo, or the Colab file upload UI), then\n# set PROJECT_DIR to wherever it ends up. Locally, the default below is fine.\nimport sys, os\nIN_COLAB = \"google.colab\" in sys.modules\nPROJECT_DIR = \"/content/sparse_concept_discovery\" if IN_COLAB else os.path.abspath(\".\")\nassert os.path.isdir(PROJECT_DIR), f\"{PROJECT_DIR} not found -- upload/sync the project there first\"\nsys.path.insert(0, PROJECT_DIR)\nif IN_COLAB:\n    %cd {PROJECT_DIR}\n    !pip install -q -r requirements.txt

In [ ]:
import torch
from src.config import CONFIG
from src import data_utils, backbone_utils, clip_grid, splice_utils, lgmd, evaluate, plotting

print("device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
splits = data_utils.load_class_splits()
{cid: {k: len(v) for k, v in s.items()} for cid, s in splits.items()}

In [ ]:
backbone = backbone_utils.Backbone()
clip_encoder = clip_grid.CLIPGridEncoder()
splice_encoder = splice_utils.SpliceEncoder()

In [ ]:
def run_class(class_id):
    grid = CONFIG["grid_size"]
    train_images = splits[class_id]["train"]
    val_images = splits[class_id]["val"]

    # --- training (factorization) set ---
    Z_train = backbone.extract_activations(train_images)
    Z_train_g = backbone_utils.downsample_to_grid(Z_train, grid)
    Abar_train = backbone_utils.unfold(Z_train_g)

    clip_train = clip_encoder.encode_images(train_images, grid_size=grid)
    S_train = splice_encoder.encode(clip_train)

    W, active_idx = lgmd.learn_basis(Abar_train, S_train, lr=CONFIG["pgd_lr"], steps=CONFIG["pgd_steps"])

    # --- held-out (inference/evaluation) set ---
    Z_val = backbone.extract_activations(val_images)
    Z_val_g = backbone_utils.downsample_to_grid(Z_val, grid)
    Abar_val = backbone_utils.unfold(Z_val_g)

    S_hat_val = lgmd.estimate_coefficients(Abar_val, W)
    Abar_hat_val = lgmd.reconstruct(S_hat_val, W)

    recon_err = evaluate.reconstruction_error(Abar_val, Abar_hat_val)
    target_idx = CONFIG["wnid_to_imagenet_idx"][class_id]
    pred_metrics = evaluate.predictive_preservation(backbone, Z_val_g, Abar_hat_val, target_idx)

    return {
        "class_id": class_id,
        "recon_err": recon_err,
        **pred_metrics,
        "W": W,
        "active_idx": active_idx,
        "S_hat_val": S_hat_val,
        "val_images": val_images,
    }

In [ ]:
from tqdm.auto import tqdm

results = {cid: run_class(cid) for cid in tqdm(CONFIG["classes"], desc="Classes")}

import os
import pandas as pd

results_df = pd.DataFrame([
    {
        "class": CONFIG["class_display_names"][cid],
        "recon_err": r["recon_err"],
        "orig_accuracy": r["orig_accuracy"],
        "recon_accuracy": r["recon_accuracy"],
        "agreement": r["agreement"],
    }
    for cid, r in results.items()
])

results_csv_path = os.path.join(CONFIG["results_dir"], "metrics.csv")
results_df.to_csv(results_csv_path, index=False)
print(f"Saved results table to {results_csv_path}")
results_df

In [ ]:
# Visualize concept heatmaps for the first 2 held-out images of each class,
# and save each figure to CONFIG["figures_dir"] (Drive-backed in Colab).
grid = CONFIG["grid_size"]
for cid, r in tqdm(results.items(), desc="Plotting concept heatmaps"):
    class_name = CONFIG["class_display_names"][cid]
    for img_idx in range(min(2, len(r["val_images"]))):
        rows = slice(img_idx * grid * grid, (img_idx + 1) * grid * grid)
        fig = plotting.plot_concept_heatmaps(
            r["val_images"][img_idx],
            r["S_hat_val"][rows],
            r["active_idx"],
            splice_encoder,
            grid_size=grid,
            top_k=CONFIG["r_concepts_to_show"],
        )
        fig_path = os.path.join(CONFIG["figures_dir"], f"{class_name}_val{img_idx}.png")
        fig.savefig(fig_path, bbox_inches="tight")
        print(f"Saved {fig_path}")